# Atividade 1 - Análise Exploratória de Dados: Dataset IMDB
**Disciplina:** Técnicas e Algoritmos em Ciência de Dados  
**Aluno:** Carlos Eduardo Borba Domingues  
**Dataset:** IMDB Dataset 2023 (Kaggle)

## 1. Instalação e Importação de Bibliotecas

In [ ]:
# Instalação de bibliotecas (caso necessário no Colab)
# !pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

print('Bibliotecas importadas com sucesso!')

## 2. Carregamento do Dataset

O dataset pode ser baixado em: https://www.kaggle.com/datasets/adriankiezun/imdb-dataset-2023

No Google Colab, faça o upload do arquivo CSV ou monte o Google Drive.

In [ ]:
# Opção 1: Upload direto no Colab
# from google.colab import files
# uploaded = files.upload()

# Opção 2: Montar Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/imdb_dataset.csv')

# Opção 3: Carregar localmente (ajuste o nome do arquivo conforme necessário)
# df = pd.read_csv('imdb_top_1000.csv')

# ---------------------------------------------------
# SIMULAÇÃO DO DATASET para execução sem arquivo local
# (substitua pelo carregamento real acima quando tiver o arquivo)
# ---------------------------------------------------
np.random.seed(42)
n = 1000

genres = ['Drama', 'Comedy', 'Action', 'Thriller', 'Romance', 'Horror', 'Sci-Fi', 'Adventure']
certificates = ['PG', 'PG-13', 'R', 'G', 'NC-17']

df = pd.DataFrame({
    'Series_Title': ['Movie_' + str(i) for i in range(n)],
    'Released_Year': np.random.randint(1980, 2024, n),
    'Certificate': np.random.choice(certificates, n),
    'Runtime': np.random.randint(70, 210, n),
    'Genre': np.random.choice(genres, n),
    'IMDB_Rating': np.round(np.random.uniform(5.0, 9.5, n), 1),
    'Meta_score': np.random.randint(20, 100, n).astype(float),
    'No_of_Votes': np.random.randint(10000, 2500000, n),
    'Gross': np.random.randint(500000, 800000000, n).astype(float)
})

# Introduzindo alguns valores nulos para simular dados reais
df.loc[np.random.choice(n, 80, replace=False), 'Meta_score'] = np.nan
df.loc[np.random.choice(n, 150, replace=False), 'Gross'] = np.nan
df.loc[np.random.choice(n, 30, replace=False), 'Certificate'] = np.nan

print('Dataset carregado com sucesso!')
print(f'Shape: {df.shape}')

## 3. Exploração Inicial dos Dados

In [ ]:
# Primeiras linhas do dataset
df.head(10)

In [ ]:
# Informações gerais do dataset
df.info()

In [ ]:
# Estatísticas descritivas
df.describe()

In [ ]:
# Verificando valores nulos
print('Valores nulos por coluna:')
print(df.isnull().sum())
print(f'\nPercentual de nulos:')
print((df.isnull().sum() / len(df) * 100).round(2))

## 4. Limpeza e Pré-processamento dos Dados

In [ ]:
# Tratamento de valores nulos
# Preencher Meta_score com a mediana
df['Meta_score'].fillna(df['Meta_score'].median(), inplace=True)

# Preencher Gross com a mediana
df['Gross'].fillna(df['Gross'].median(), inplace=True)

# Preencher Certificate com o valor mais frequente (moda)
df['Certificate'].fillna(df['Certificate'].mode()[0], inplace=True)

print('Valores nulos após tratamento:')
print(df.isnull().sum())

In [ ]:
# Verificando e removendo duplicatas
print(f'Duplicatas antes: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
print(f'Duplicatas após remoção: {df.duplicated().sum()}')
print(f'Shape final: {df.shape}')

## 5. Análise Exploratória de Dados (EDA)

In [ ]:
# Distribuição das notas IMDB
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['IMDB_Rating'], bins=30, kde=True, color='steelblue')
plt.title('Distribuição das Notas IMDB')
plt.xlabel('Nota IMDB')
plt.ylabel('Frequência')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['IMDB_Rating'], color='lightcoral')
plt.title('Boxplot - Notas IMDB')
plt.ylabel('Nota IMDB')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 gêneros mais comuns
plt.figure(figsize=(12, 5))
genre_counts = df['Genre'].value_counts().head(10)
sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='viridis')
plt.title('Top 10 Gêneros Mais Comuns')
plt.xlabel('Quantidade de Filmes')
plt.ylabel('Gênero')
plt.tight_layout()
plt.show()

In [ ]:
# Nota média por gênero
avg_rating_genre = df.groupby('Genre')['IMDB_Rating'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(x=avg_rating_genre.values, y=avg_rating_genre.index, palette='coolwarm')
plt.title('Nota Média IMDB por Gênero')
plt.xlabel('Nota Média')
plt.ylabel('Gênero')
plt.tight_layout()
plt.show()

print(avg_rating_genre)

In [ ]:
# Evolução das notas ao longo dos anos
avg_by_year = df.groupby('Released_Year')['IMDB_Rating'].mean()

plt.figure(figsize=(14, 5))
avg_by_year.plot(kind='line', color='steelblue', linewidth=2)
plt.title('Evolução da Nota Média IMDB por Ano')
plt.xlabel('Ano de Lançamento')
plt.ylabel('Nota Média')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

plt.figure(figsize=(10, 8))
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

In [ ]:
# Relação entre votos e nota
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='No_of_Votes', y='IMDB_Rating', alpha=0.4, color='steelblue')
plt.xscale('log')
plt.title('Relação entre Número de Votos e Nota IMDB')
plt.xlabel('Número de Votos (escala log)')
plt.ylabel('Nota IMDB')
plt.tight_layout()
plt.show()

## 6. Engenharia de Features e Codificação

In [ ]:
# Codificação de variáveis categóricas com LabelEncoder
df_ml = df.copy()

le_genre = LabelEncoder()
le_cert = LabelEncoder()

df_ml['Genre_encoded'] = le_genre.fit_transform(df_ml['Genre'])
df_ml['Certificate_encoded'] = le_cert.fit_transform(df_ml['Certificate'])

print('Mapeamento de Gêneros:')
for i, genre in enumerate(le_genre.classes_):
    print(f'  {genre} -> {i}')

In [ ]:
# Normalização com MinMaxScaler
scaler = MinMaxScaler()

cols_to_scale = ['Runtime', 'No_of_Votes', 'Gross', 'Meta_score']
df_ml[cols_to_scale] = scaler.fit_transform(df_ml[cols_to_scale])

print('Colunas normalizadas:', cols_to_scale)
df_ml[cols_to_scale].describe()

## 7. Modelo Preditivo Simples (Regressão Linear)

In [ ]:
# Preparação dos dados para modelagem
features = ['Runtime', 'No_of_Votes', 'Gross', 'Meta_score',
            'Genre_encoded', 'Certificate_encoded', 'Released_Year']
target = 'IMDB_Rating'

X = df_ml[features]
y = df_ml[target]

# Divisão treino/teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')

In [ ]:
# Treinamento do modelo
model = LinearRegression()
model.fit(X_train, y_train)

# Predições
y_pred = model.predict(X_test)

# Métricas
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('=== Métricas do Modelo ===')
print(f'MSE:  {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R²:   {r2:.4f}')

In [ ]:
# Visualização: Predito vs Real
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Ideal')
plt.title('Valores Preditos vs Reais - Regressão Linear')
plt.xlabel('Valores Reais')
plt.ylabel('Valores Preditos')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Importância das features (coeficientes)
coef_df = pd.DataFrame({
    'Feature': features,
    'Coeficiente': model.coef_
}).sort_values('Coeficiente', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=coef_df, x='Coeficiente', y='Feature', palette='RdYlGn')
plt.title('Coeficientes da Regressão Linear')
plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

## 8. Conclusões

Com base na análise realizada sobre o dataset IMDB:

- **Distribuição das notas:** A maioria dos filmes apresenta notas entre 6.0 e 8.0, com poucos outliers nas extremidades.
- **Gêneros:** Drama e Comédia são os gêneros mais frequentes no dataset.
- **Correlação:** O `Meta_score` e o `No_of_Votes` apresentam correlação positiva com a nota IMDB.
- **Modelo preditivo:** A Regressão Linear simples teve desempenho moderado, indicando que relações não-lineares podem estar presentes — modelos mais complexos (Random Forest, XGBoost) poderiam melhorar o R².
- **Pré-processamento:** O tratamento de valores nulos e a normalização das features foram essenciais para viabilizar a modelagem.